In [1]:
import pandas as pd
df = pd.read_csv("data/appointments.csv")
print(df.shape)

(1020, 14)


# Task 1.1 - Read with csv module

In [2]:
import csv
import json


def parse_row(row):
  parsed = row.copy()
  parsed["age"] = int(row["age"])
  parsed["fee_inr"] = int(row["fee_inr"])
  parsed["prior_no_shows"] = int(row["prior_no_shows"])
  parsed["distance_km"] = float(row["distance_km"])
  return parsed


accepted = []
rejected = []
total = 0

with open("data/appointments.csv", newline="") as f:
  reader = csv.DictReader(f)
  for row in reader:
    total += 1
    try:
      accepted.append(parse_row(row))
    except ValueError:
      rejected.append(row)

print("Rows read    :", total)
print("Accepted     :", len(accepted))
print("Rejected     :", len(rejected))

with open("rejected_rows.json", "w") as f:
  json.dump(rejected, f, indent=2)

Rows read    : 1020
Accepted     : 988
Rejected     : 32


In [3]:
missing_age = sum(1 for r in rejected if r["age"].strip() == "")
missing_dist = sum(1 for r in rejected if r["distance_km"].strip() == "")
print("Rejected with missing age     :", missing_age)
print("Rejected with missing distance:", missing_dist)

Rejected with missing age     : 12
Rejected with missing distance: 20


# Task 1.2 - Inspect and Clean, Pass 1

In [4]:
import pandas as pd

df = pd.read_csv("data/appointments.csv")

print("Shape:", df.shape)
df.info()
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
a

In [5]:
print("Start rows:", len(df))

df = df.drop_duplicates()
print("After dropping duplicates:", len(df))

df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(df["booking_channel"].unique())
print("Distinct channels:", df["booking_channel"].nunique())
print("After cleaning booking_channel:", len(df))

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After date conversion:", len(df))

df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing appt_date/status:", len(df))
print("Final shape:", df.shape)

Start rows: 1020
After dropping duplicates: 1000
<StringArray>
['Phone', 'Walk-In', 'App']
Length: 3, dtype: str
Distinct channels: 3
After cleaning booking_channel: 1000
After date conversion: 1000
After dropping missing appt_date/status: 990
Final shape: (990, 14)


# Task 1.3 - Clean, Pass 2

In [6]:
import numpy as np

invalid_age = ((df["age"] < 1) | (df["age"] > 100)).sum()
print("Invalid ages:", invalid_age)

df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan
print("Missing age total:", df["age"].isnull().sum())
print("Missing distance_km:", df["distance_km"].isnull().sum())

median_age = int(round(df["age"].median()))
median_dist = df["distance_km"].median()
print("Median age:", median_age)
print("Median distance_km:", median_dist)

df["age"] = df["age"].fillna(median_age).astype(int)
df["distance_km"] = df["distance_km"].fillna(median_dist)
print("Missing age after fill:", df["age"].isnull().sum())
print("Missing distance after fill:", df["distance_km"].isnull().sum())

Invalid ages: 5
Missing age total: 17
Missing distance_km: 20
Median age: 48
Median distance_km: 8.25
Missing age after fill: 0
Missing distance after fill: 0


In [7]:
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_capped = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(col, "-> Q1:", q1, "Q3:", q3, "IQR:", iqr)
    print(col, "-> lower:", lower, "upper:", upper)
    print(col, "-> capped values:", n_capped)

df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Max fee after capping:", df["fee_inr"].max())
print("Final shape:", df.shape)
print(df.isnull().sum())

distance_km -> Q1: 4.625 Q3: 13.0 IQR: 8.375
distance_km -> lower: -7.9375 upper: 25.5625
distance_km -> capped values: 26
fee_inr -> Q1: 500.0 Q3: 787.5 IQR: 287.5
fee_inr -> lower: 68.75 upper: 1218.75
fee_inr -> capped values: 3
Max fee after capping: 1219
Final shape: (990, 14)
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
dtype: int64


# Task 1.4 - Features and NumPy Summary

In [8]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(df["status"].value_counts())
print("lead_days min:", df["lead_days"].min())
print("lead_days max:", df["lead_days"].max())
print("No-shows:", df["no_show"].sum(), "out of", len(df))
print("No-show percentage:", round(df["no_show"].mean() * 100, 1))

status
Attended    772
No-Show     218
Name: count, dtype: int64
lead_days min: 1
lead_days max: 45
No-shows: 218 out of 990
No-show percentage: 22.0


In [9]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("X shape:", X.shape)

col_means = X.mean(axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:")
print(np.round(col_means, 2))

mask = X[:, 3] > 14
print("Rows with lead_days > 14:", mask.sum())
print("Percentage with lead_days > 14:", round(mask.mean() * 100, 1))

x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("Rows with |z| > 2 on distance_km:", (np.abs(z) > 2).sum())

X shape: (990, 4)
Column means [age, distance_km, fee_inr, lead_days]:
[ 48.17   9.5  635.71  11.82]
Rows with lead_days > 14: 296
Percentage with lead_days > 14: 29.9
Rows with |z| > 2 on distance_km: 53


# Task 1.5 - Enrich Clinics via Live Weather API

In [10]:
import requests

clinics = pd.read_csv("data/clinics.csv")
print(clinics.shape)
print(clinics)

(8, 6)
  clinic_id               clinic_name     city    tier  latitude  longitude
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462


In [11]:
def get_temp(lat, lon, name):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    reason = "unknown"
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)
        except requests.exceptions.RequestException as e:
            reason = "request error: " + str(e)
            continue
        if response.status_code != 200:
            reason = "status code " + str(response.status_code)
            break
        try:
            return response.json()["hourly"]["temperature_2m"][0]
        except (KeyError, IndexError, ValueError):
            reason = "hourly data missing"
            break
    print("  FAILED:", name, "-", reason)
    return None

temps = []
for _, row in clinics.iterrows():
    t = get_temp(row["latitude"], row["longitude"], row["clinic_name"])
    print(row["clinic_name"], "(", row["city"], ") ->", t)
    temps.append(t)

clinics["temp_c"] = temps
print(clinics[["clinic_name", "city", "temp_c"]])

Kothrud Family Clinic ( Pune ) -> 17.7
Hadapsar Care Centre ( Pune ) -> 18.8
Sitabuldi Health Point ( Nagpur ) -> 20.8
Malviya Nagar Clinic ( Jaipur ) -> 19.8
Vaishali Nagar Clinic ( Jaipur ) -> 19.8
Edappally Medical Centre ( Kochi ) -> 25.5
Vijay Nagar Clinic ( Indore ) -> 18.7
Gomti Nagar Care Point ( Lucknow ) -> 18.6
                clinic_name     city  temp_c
0     Kothrud Family Clinic     Pune    17.7
1      Hadapsar Care Centre     Pune    18.8
2    Sitabuldi Health Point   Nagpur    20.8
3      Malviya Nagar Clinic   Jaipur    19.8
4     Vaishali Nagar Clinic   Jaipur    19.8
5  Edappally Medical Centre    Kochi    25.5
6        Vijay Nagar Clinic   Indore    18.7
7    Gomti Nagar Care Point  Lucknow    18.6


In [12]:
clinics.to_csv("clinics_enriched.csv", index=False)

check = pd.read_csv("clinics_enriched.csv")
print(check.shape)
print(check.columns.tolist())
print(check.isnull().sum())
print(check)

(8, 7)
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']
clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
temp_c         0
dtype: int64
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.7  
1    18.8  
2    20.8  
3    19.8  
4    19.8

# Task 1.6 - Export and SQL load script

In [14]:
final_cols = ["appointment_id", "patient_id", "clinic_id", "booked_on",
              "appt_date", "age", "gender", "distance_km", "fee_inr",
              "booking_channel", "visit_type", "reminder_sent",
              "prior_no_shows", "status", "lead_days", "no_show"]

out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[final_cols]

print("Shape:", out.shape)
print("Distinct appointment_id:", out["appointment_id"].nunique())
print("appointment_id unique:", out["appointment_id"].is_unique)
print("Total missing values:", out.isnull().sum().sum())
print(out.isnull().sum())
print(out.head(3))

Shape: (990, 16)
Distinct appointment_id: 990
appointment_id unique: True
Total missing values: 0
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850        

In [15]:
out.to_csv("appointments_clean.csv", index=False)

check = pd.read_csv("appointments_clean.csv")
print(check.shape)
print(check.columns.tolist())

(990, 16)
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [16]:
def sql_value(v):
    if pd.isna(v):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return str(v)

clinics_e = pd.read_csv("clinics_enriched.csv")
lines = []

for row in clinics_e.itertuples(index=False):
    values = ",".join(sql_value(v) for v in row)
    lines.append(f"INSERT INTO clinics VALUES ({values});")

for row in out.itertuples(index=False):
    values = ",".join(sql_value(v) for v in row)
    lines.append(f"INSERT INTO appointments VALUES ({values});")

with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print("Total statements:", len(lines))
print("Clinics statements:", sum(l.startswith("INSERT INTO clinics") for l in lines))
print("Appointments statements:", sum(l.startswith("INSERT INTO appointments") for l in lines))
print(lines[0])
print(lines[8])

Total statements: 998
Clinics statements: 8
Appointments statements: 990
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,17.7);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);


# Part 2 - Task 2.2 Python groupby check

In [1]:
import pandas as pd

appts = pd.read_csv("appointments_clean.csv")
clin = pd.read_csv("clinics_enriched.csv")

merged = appts.merge(clin[["clinic_id", "clinic_name", "city"]], on="clinic_id")
g = (merged.groupby(["clinic_name", "city"])
           .agg(appointments=("appointment_id", "count"),
                no_shows=("no_show", "sum"))
           .reset_index())
g["no_show_pct"] = (100 * g["no_shows"] / g["appointments"]).round(1)
print(g.sort_values("no_show_pct", ascending=False).to_string(index=False))
print("Total appointments:", g["appointments"].sum())
print("Total no-shows:", g["no_shows"].sum())

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
Total appointments: 990
Total no-shows: 218
